# Hybrid Retriever and Combing Dense and sparse Retriever

In [1]:
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain.retrievers import BM25Retriever
from langchain.retrievers import EnsembleRetriever
from langchain.schema import Document



In [2]:
##sample data

docs = [
    Document(page_content="RAG combines retrieval and generation."),
    Document(page_content="BMW is headquartered in Munich."),
    Document(page_content="ChromaDB is a vector database."),
    Document(page_content="BMW was founded in 1916."),
    Document(page_content="Groq builds AI inference hardware.")
]

embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
dense_vectorestore = FAISS.from_documents(docs,embedding_model)
dense_retriever = dense_vectorestore.as_retriever()

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 13475.99it/s]


In [4]:
##Sparse Retriever (BM25)
sparse_retriever = BM25Retriever.from_documents(docs)
sparse_retriever.k=3 ## top k result




In [5]:
## cobine the both of them ( dense + sparse ) with ensemble retriever

hybrid_retriever = EnsembleRetriever(
    retrievers=[dense_retriever,sparse_retriever],
    weights=[0.7,0.3]
)

In [6]:
hybrid_retriever

EnsembleRetriever(retrievers=[VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x12513f010>, search_kwargs={}), BM25Retriever(vectorizer=<rank_bm25.BM25Okapi object at 0x109df1550>, k=3)], weights=[0.7, 0.3])

In [8]:
query = "What is BMW ?"

result = hybrid_retriever.invoke(query)

for i,doc in enumerate(result):
    print(f"\n Document : {i+1}. \n Chunk : {doc.page_content}")


 Document : 1. 
 Chunk : BMW is headquartered in Munich.

 Document : 2. 
 Chunk : BMW was founded in 1916.

 Document : 3. 
 Chunk : ChromaDB is a vector database.

 Document : 4. 
 Chunk : Groq builds AI inference hardware.


# RAG Pipeline

In [11]:
from langchain_groq import ChatGroq
from langchain.prompts import PromptTemplate
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain.chains .retrieval import create_retrieval_chain

In [14]:
prompt = PromptTemplate.from_template(
    """Anwer the question based on the context below.
    Context:
    {context}
    
    Question: {input}
    """
)

In [18]:
from dotenv import load_dotenv
# Load environment variables
load_dotenv()
import os

# Initialize LLM
llm = ChatGroq(
    groq_api_key=os.getenv("GROQ_API_KEY"),
    model_name="qwen/qwen3-32b"
)

# Invoke model
response = llm.invoke("Hi")
print(response)

content='<think>\nOkay, the user just said "Hi". That\'s pretty open-ended. I need to figure out how to respond appropriately. Let me start by acknowledging their greeting. Since there\'s no specific question or topic mentioned, I should probably offer help in a general way.\n\nMaybe they need assistance with something specific, so I can ask if they have any particular questions or if there\'s something I can assist them with. Keeping the tone friendly and approachable is key here. I don\'t want to assume what they need, so it\'s best to invite them to specify. Let me make sure the response is concise and not too verbose. Alright, time to put that together.\n</think>\n\nHello! How can I assist you today? Feel free to ask if you have any questions or need help with anything specific. 😊' additional_kwargs={} response_metadata={'token_usage': {'completion_tokens': 166, 'prompt_tokens': 9, 'total_tokens': 175, 'completion_time': 0.374382569, 'prompt_time': 0.000380354, 'queue_time': 0.0634

In [19]:
##create a stuff document chain
document_chain = create_stuff_documents_chain(llm=llm,prompt=prompt)

#create full rag chain
rag_chain = create_retrieval_chain(retriever=hybrid_retriever,combine_docs_chain=document_chain)

rag_chain


RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableBinding(bound=RunnableLambda(lambda x: x['input'])
           | EnsembleRetriever(retrievers=[VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x12513f010>, search_kwargs={}), BM25Retriever(vectorizer=<rank_bm25.BM25Okapi object at 0x109df1550>, k=3)], weights=[0.7, 0.3]), kwargs={}, config={'run_name': 'retrieve_documents'}, config_factories=[])
})
| RunnableAssign(mapper={
    answer: RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
              context: RunnableLambda(format_docs)
            }), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
            | PromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, template='Anwer the question based on the context below.\n    Context:\n    {context}\n\n    Question: {input}\n    ')
            | ChatGroq(client=<groq

In [21]:
#question
query = {"input": "what is BMW ?"}
response = rag_chain.invoke(query)


#output

print (f"The answer : {response['answer']}")

print("Source of the document ")

for i,doc in enumerate(response['context']):
    print(f"Docs : { i+1} \n {doc.page_content}")

The answer : <think>
Okay, I need to answer the question "What is BMW?" based on the given context. Let me look at the context again.

The context says that BMW is headquartered in Munich and was founded in 1916. The other two points are about ChromaDB and Groq, which don't seem relevant here. So, the key information is the location and founding year. 

I remember that BMW is a well-known car manufacturer, but the context doesn't mention that directly. However, since the user provided only the headquarters and founding date, I should stick to that. Wait, but maybe the user expects more? Hmm, the instructions say to base the answer strictly on the context. So I can't add extra knowledge. 

So the answer should state that BMW is a company headquartered in Munich, founded in 1916. But the user might want to know what kind of company BMW is. Since the context doesn't specify, I can't assume. Maybe just present the facts given: location and founding year. If the user expects more, but the c